# RQ2 Secondary Calibration — Sensitivity Analysis

**This notebook is a sensitivity analysis only.**
The uncalibrated Group B1 result (P=0.049, R=0.288, F1=0.080) is the **primary reported result**.
This analysis is labelled *B1c / B1s (Sensitivity Analysis)* throughout.

### What this notebook does
1. Defines the fixed canonical vocabulary **V** (49 concept labels from Emile's vault;
   6 bibliographic paper-title tags excluded).
2. Embeds V with `nomic-ai/nomic-embed-text-v1.5` (`clustering:` prefix) and **freezes
   τ_m = p90** of intra-vocabulary pairwise cosines — before any agent label is seen.
3. Splits 22 papers into a **7-paper dev set** and **15-paper test set** (stratified, deterministic).
4. Adds a **same-subset uncalibrated baseline** (hasTopic only, 15 test papers, no mapping)
   so B1c can be fairly compared.
5. Grid-searches deduplication threshold τ_d and top-k on the dev set only.
6. Applies the frozen pipeline to the held-out test set.
7. Reports:
   - **B1-hasTopic-15** — uncalibrated hasTopic-only baseline on same 15 papers (fair comparison)
   - **B1c** — calibrated exact Precision / Recall / F1
   - **B1s** — cosine proximity upper bound (exploratory, sensitive to threshold choice)
   - Sensitivity checks at 0.75 and 0.80 as footnotes only.


In [37]:
import json, re, os
from pathlib import Path
from itertools import combinations

import numpy as np
import yaml
from sentence_transformers import SentenceTransformer

# Load .env
env_path = Path(".env")
if env_path.exists():
    for line in env_path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            k, v = line.split("=", 1)
            os.environ[k.strip()] = v.strip().strip("'").strip('"')

print("Imports OK.")


Imports OK.


In [38]:
GOLD_META   = Path("data/gold_labels/emile_vault_gold.json")
VAULT_ARXIV = Path("data/gold_labels/emile_vault_arxiv_ids.json")
NOTES_DIR   = Path("data/generated_notes")

with open(GOLD_META,   encoding="utf-8") as f: gold_meta   = json.load(f)
with open(VAULT_ARXIV, encoding="utf-8") as f: vault_arxiv = json.load(f)

arxiv_to_title = {v: k for k, v in vault_arxiv.items()}
overlap_ids    = sorted(arxiv_to_title.keys())
assert len(overlap_ids) == 22, f"Expected 22 overlap papers, got {len(overlap_ids)}"
print(f"Loaded {len(overlap_ids)} overlap papers.")


Loaded 22 overlap papers.


In [39]:
# Paper-title tags to exclude — bibliographic references, not concept labels
PAPER_TITLE_TAGS = {
    "Flow Matching for Generative Modeling",
    "Flow Matching_ Matching flows instead of scores",
    "Mamba_ Linear-Time Sequence Modeling with Selective State Spaces",
    "Sigmoid loss for language image pre-training",
    "Learning transferable visual models from natural language supervision",
    "Bridging Machine Learning and Logical Reasoning by Abductive Learning",
}

all_emile_tags = set()
for title, arxiv_id in vault_arxiv.items():
    tags = gold_meta.get(title, {}).get("topics", [])
    all_emile_tags.update(tags)

V = sorted(all_emile_tags - PAPER_TITLE_TAGS)

print(f"Full Emile vocabulary:  {len(all_emile_tags)} labels")
print(f"Paper-title tags removed: {len(all_emile_tags) - len(V)}")
print(f"Canonical vocabulary |V|: {len(V)} concept labels")
print()
for i, label in enumerate(V, 1):
    print(f"  [{i:2d}] {label}")


Full Emile vocabulary:  55 labels
Paper-title tags removed: 6
Canonical vocabulary |V|: 49 concept labels

  [ 1] Boolean circuit
  [ 2] Laplace approximation
  [ 3] abductive reasoning
  [ 4] chain-of-thought
  [ 5] controlled generation
  [ 6] deep generative models
  [ 7] dependent type theory
  [ 8] differentiable logics
  [ 9] differentiable probabilistic logics
  [10] diffusion models
  [11] discrete diffusion
  [12] discrete diffusion models
  [13] discrete latent structures
  [14] dynamic programming
  [15] ensembles
  [16] expressivity of models
  [17] fuzzy logic
  [18] generalisation
  [19] generative modeling
  [20] graph neural network
  [21] homophily
  [22] importance sampling
  [23] inductive logic programming
  [24] large language model reasoning
  [25] latent variable models
  [26] masked language models
  [27] modality alignment
  [28] neurosymbolic learning
  [29] neurosymbolic verification
  [30] node classification
  [31] on policy RL
  [32] parallelization
  [33]

In [40]:
MODEL_NAME = "nomic-ai/nomic-embed-text-v1.5"
PREFIX     = "clustering: "   # symmetric label-to-label similarity — NOT search_document

print(f"Loading {MODEL_NAME} ...")
model = SentenceTransformer(MODEL_NAME, trust_remote_code=True)

# Encode canonical vocabulary
v_texts  = [PREFIX + label for label in V]
E_V_raw  = model.encode(v_texts, normalize_embeddings=True, show_progress_bar=False)
E_V      = E_V_raw  # already L2-normalised by normalize_embeddings=True

# Pairwise cosine matrix (dot product of unit vectors)
C = E_V @ E_V.T  # (50, 50)

# Extract upper triangle, excluding diagonal
upper_idx  = np.triu_indices(len(V), k=1)
pairwise   = C[upper_idx]

# Descriptive statistics
percentiles = {f"p{p}": float(np.percentile(pairwise, p))
               for p in [10, 25, 50, 75, 90, 95]}
tau_m = float(np.percentile(pairwise, 90))

threshold_log = {
    "vocabulary_size":      len(V),
    "n_pairs":              int(len(pairwise)),
    "cosine_distribution": {
        "min":    float(pairwise.min()),
        **percentiles,
        "max":    float(pairwise.max()),
    },
    "tau_m":   tau_m,
    "status":  "FROZEN — not adjusted after this point",
}

# Save log
out_path = Path("data/results/b1c_vocab_threshold.json")
out_path.parent.mkdir(parents=True, exist_ok=True)
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(threshold_log, f, indent=2)

print("=" * 55)
print(f"Intra-vocabulary cosine distribution ({len(pairwise)} pairs):")
for k, val in percentiles.items():
    marker = " <-- τ_m (FROZEN)" if k == "p90" else ""
    print(f"  {k}: {val:.4f}{marker}")
print("=" * 55)
print(f"tau_m = {tau_m:.4f}  (saved to {out_path})")


Loading nomic-ai/nomic-embed-text-v1.5 ...


<All keys matched successfully>


Intra-vocabulary cosine distribution (1176 pairs):
  p10: 0.6395
  p25: 0.6598
  p50: 0.6826
  p75: 0.7067
  p90: 0.7361 <-- τ_m (FROZEN)
  p95: 0.7759
tau_m = 0.7361  (saved to data\results\b1c_vocab_threshold.json)


In [41]:
# Build per-paper metadata for splitting
paper_info = []
for arxiv_id in overlap_ids:
    title     = arxiv_to_title[arxiv_id]
    emile_tags = gold_meta.get(title, {}).get("topics", [])
    paper_info.append({"arxiv_id": arxiv_id, "title": title,
                       "n_emile": len(emile_tags), "emile_tags": emile_tags})

# Stratified split by n_emile, deterministic within stratum (sorted arXiv ID)
from collections import defaultdict
strata = defaultdict(list)
for p in paper_info:
    strata[p["n_emile"]].append(p["arxiv_id"])
for n in strata:
    strata[n].sort()

dev_ids, test_ids = [], []
for n in sorted(strata.keys(), reverse=True):
    ids = strata[n]
    n_dev = len(ids) // 3
    dev_ids.extend(ids[:n_dev])
    test_ids.extend(ids[n_dev:])

print(f"Dev  set: {len(dev_ids)} papers  -> {dev_ids}")
print(f"Test set: {len(test_ids)} papers -> {test_ids}")
assert len(dev_ids) + len(test_ids) == 22


Dev  set: 7 papers  -> ['2412.06014', '2410.02143', '1711.11157', '2212.12393', '2312.04474', '2012.13635', '2305.19951']
Test set: 15 papers -> ['2310.17451', '2505.23061', '2507.14484', '2602.23878', '2603.03612', '2401.05224', '2402.12240', '2502.03274', '2505.13138', '2507.21053', '2602.17270', '2506.03719', '2509.26625', '2502.01341', '2510.14538']


In [42]:
def normalise(s):
    "Lowercase, strip punctuation, collapse whitespace."
    if not isinstance(s, str):
        s = str(s or "")
    s = re.sub(r"\[\[([^\]]+)\]\]", lambda m: m.group(1).split("|")[0], s)
    return re.sub(r"[^a-z0-9 ]", "", s.lower()).strip()

def load_agent_topics(arxiv_id):
    "Read hasTopic from the generated note frontmatter."
    nf = NOTES_DIR / f"{arxiv_id}.md"
    content = nf.read_text(encoding="utf-8", errors="ignore")
    fm_match = re.match(r"^---\s*\n(.*?)\n---\s*\n?(.*)", content, re.DOTALL)
    if not fm_match:
        return []
    try:
        fm = yaml.safe_load(fm_match.group(1)) or {}
    except Exception:
        return []
    topics = fm.get("hasTopic", []) or []
    if isinstance(topics, str):
        topics = [topics]
    return [t for t in topics if t]

def encode_labels(labels):
    "Encode a list of label strings with the Nomic prefix, L2-normalised."
    if not labels:
        return np.zeros((0, E_V.shape[1]))
    texts = [PREFIX + lbl for lbl in labels]
    return model.encode(texts, normalize_embeddings=True, show_progress_bar=False)

def deduplicate(labels, embeddings, tau_d):
    "Remove near-duplicate agent labels. Keep one with best canonical sim."
    if len(labels) <= 1:
        return labels, embeddings
    kept = list(range(len(labels)))
    sims_to_canon = (embeddings @ E_V.T).max(axis=1)  # best sim to any V label
    removed = set()
    for i in range(len(labels)):
        if i in removed:
            continue
        for j in range(i + 1, len(labels)):
            if j in removed:
                continue
            if float(embeddings[i] @ embeddings[j]) > tau_d:
                # remove the one with lower canonical similarity
                if sims_to_canon[i] >= sims_to_canon[j]:
                    removed.add(j)
                else:
                    removed.add(i)
                    break
    keep_idx = [i for i in range(len(labels)) if i not in removed]
    return [labels[i] for i in keep_idx], embeddings[keep_idx]

def map_and_topk(labels, embeddings, tau_m, k, mapping_log=None, arxiv_id=None):
    "Map agent labels to canonical V labels, discard below tau_m, keep top-k."
    if len(labels) == 0:
        return [], []
    sim_matrix = embeddings @ E_V.T  # (n_agent, 50)
    best_idx   = sim_matrix.argmax(axis=1)
    best_sim   = sim_matrix.max(axis=1)

    canonical_labels = []
    canonical_sims   = []
    for i, (lbl, b_idx, b_sim) in enumerate(zip(labels, best_idx, best_sim)):
        decision = "MAPPED" if b_sim >= tau_m else "DISCARDED"
        if mapping_log is not None:
            mapping_log.append({
                "paper":               arxiv_id,
                "original_agent_label": lbl,
                "best_canonical_match": V[int(b_idx)],
                "cosine_similarity":    round(float(b_sim), 4),
                "tau_m":               round(tau_m, 4),
                "decision":            decision,
            })
        if decision == "MAPPED":
            canonical_labels.append(V[int(b_idx)])
            canonical_sims.append(float(b_sim))

    # Deduplicate canonical labels (same canonical target from different sources)
    seen, deduped_labels, deduped_sims = set(), [], []
    for lbl, sim in sorted(zip(canonical_labels, canonical_sims),
                            key=lambda x: -x[1]):
        if lbl not in seen:
            seen.add(lbl)
            deduped_labels.append(lbl)
            deduped_sims.append(sim)

    # Top-k retention
    return deduped_labels[:k], deduped_sims[:k]

def exact_prf(pred_labels, gold_labels):
    pred_set = {normalise(l) for l in pred_labels if normalise(l)}
    gold_set = {normalise(l) for l in gold_labels if normalise(l)}
    if not gold_set:
        return dict(p=0.0, r=0.0, f1=0.0, tp=0, n_pred=len(pred_set), n_gold=0)
    tp = len(pred_set & gold_set)
    prec = tp / len(pred_set) if pred_set else 0.0
    rec  = tp / len(gold_set)
    f1   = 2*prec*rec / (prec+rec) if (prec+rec) > 0 else 0.0
    return dict(p=round(prec,4), r=round(rec,4), f1=round(f1,4),
                tp=tp, n_pred=len(pred_set), n_gold=len(gold_set))

def soft_prf(pred_labels, gold_labels):
    "Continuous cosine partial credit — no binary threshold."
    if not pred_labels or not gold_labels:
        return dict(p=0.0, r=0.0, f1=0.0)
    E_pred = encode_labels(pred_labels)
    E_gold = encode_labels(gold_labels)
    sim    = E_pred @ E_gold.T  # (n_pred, n_gold)
    soft_p = float(sim.max(axis=1).mean())   # each pred: best match to any gold
    soft_r = float(sim.max(axis=0).mean())   # each gold: best match to any pred
    soft_f = 2*soft_p*soft_r / (soft_p+soft_r) if (soft_p+soft_r) > 0 else 0.0
    return dict(p=round(soft_p,4), r=round(soft_r,4), f1=round(soft_f,4))

print("Helper functions defined.")


Helper functions defined.


In [43]:
k_grid     = [3, 4, 5]
tau_d_grid = [0.85, 0.90, 0.95]

dev_grid_results = []

for tau_d in tau_d_grid:
    for k in k_grid:
        paper_f1s = []
        for arxiv_id in dev_ids:
            title      = arxiv_to_title[arxiv_id]
            gold_tags  = gold_meta.get(title, {}).get("topics", [])
            # Remove paper-title tags from gold for fair comparison
            gold_tags  = [t for t in gold_tags if t not in PAPER_TITLE_TAGS]
            agent_lbls = load_agent_topics(arxiv_id)
            if not agent_lbls:
                paper_f1s.append(0.0)
                continue
            E_agent = encode_labels(agent_lbls)
            dedup_lbls, dedup_embs = deduplicate(agent_lbls, E_agent, tau_d)
            canon_lbls, _ = map_and_topk(dedup_lbls, dedup_embs, tau_m, k)
            metrics = exact_prf(canon_lbls, gold_tags)
            paper_f1s.append(metrics["f1"])
        macro_f1 = round(float(np.mean(paper_f1s)), 4)
        dev_grid_results.append({"tau_d": tau_d, "k": k, "dev_macro_f1": macro_f1})
        print(f"  tau_d={tau_d}  k={k}  dev_macro_F1={macro_f1:.4f}")

# Select best
best = max(dev_grid_results, key=lambda x: x["dev_macro_f1"])
best_tau_d = best["tau_d"]
best_k     = best["k"]
print(f"\n==> Best on dev: tau_d={best_tau_d}  k={best_k}  F1={best['dev_macro_f1']:.4f}")
print("Hyperparameters FROZEN. Test set will be opened exactly once.")

Path("data/results/b1c_dev_grid.json").write_text(
    json.dumps({"grid": dev_grid_results, "best": best}, indent=2), encoding="utf-8")


  tau_d=0.85  k=3  dev_macro_F1=0.3292
  tau_d=0.85  k=4  dev_macro_F1=0.3394
  tau_d=0.85  k=5  dev_macro_F1=0.3213
  tau_d=0.9  k=3  dev_macro_F1=0.3292
  tau_d=0.9  k=4  dev_macro_F1=0.3231
  tau_d=0.9  k=5  dev_macro_F1=0.3050
  tau_d=0.95  k=3  dev_macro_F1=0.3292
  tau_d=0.95  k=4  dev_macro_F1=0.3231
  tau_d=0.95  k=5  dev_macro_F1=0.3050

==> Best on dev: tau_d=0.85  k=4  F1=0.3394
Hyperparameters FROZEN. Test set will be opened exactly once.


781

In [44]:
# Uncalibrated hasTopic-only baseline on the same 15 test papers.
# Paper-title tags removed from gold, but NO mapping, dedup, or top-k applied to agent tags.
# This is the fair direct comparison for B1c.

b1_ht15_results = []
for arxiv_id in test_ids:
    title     = arxiv_to_title[arxiv_id]
    gold_tags = gold_meta.get(title, {}).get("topics", [])
    gold_tags = [t for t in gold_tags if t not in PAPER_TITLE_TAGS]
    agent_lbls = load_agent_topics(arxiv_id)   # hasTopic only, no processing
    m = exact_prf(agent_lbls, gold_tags)
    b1_ht15_results.append({
        "arxiv_id": arxiv_id,
        "title":    title,
        "metrics":  m,
        "n_agent":  len(agent_lbls),
        "n_gold":   len(gold_tags),
    })

def macro_ht15(key):
    return round(float(np.mean([r["metrics"][key] for r in b1_ht15_results])), 4)

print("=" * 55)
print("B1-hasTopic-15  (uncalibrated, same 15 test papers)")
print("hasTopic only · paper-title tags removed · no mapping")
print("=" * 55)
print(f"  Precision : {macro_ht15('p'):.4f}")
print(f"  Recall    : {macro_ht15('r'):.4f}")
print(f"  F1        : {macro_ht15('f1'):.4f}")
print(f"  Avg agent tags : {np.mean([r['n_agent'] for r in b1_ht15_results]):.1f}")
print(f"  Avg gold tags  : {np.mean([r['n_gold']  for r in b1_ht15_results]):.1f}")
print()
print("This is the direct comparison point for B1c (same subset, same input field).")


B1-hasTopic-15  (uncalibrated, same 15 test papers)
hasTopic only · paper-title tags removed · no mapping
  Precision : 0.1006
  Recall    : 0.1989
  F1        : 0.1256
  Avg agent tags : 5.3
  Avg gold tags  : 2.9

This is the direct comparison point for B1c (same subset, same input field).


In [45]:
mapping_log      = []
b1c_test_results = []
b1s_test_results = []

for arxiv_id in test_ids:
    title     = arxiv_to_title[arxiv_id]
    gold_tags = gold_meta.get(title, {}).get("topics", [])
    gold_tags = [t for t in gold_tags if t not in PAPER_TITLE_TAGS]

    agent_lbls = load_agent_topics(arxiv_id)
    E_agent    = encode_labels(agent_lbls)

    dedup_lbls, dedup_embs = deduplicate(agent_lbls, E_agent, best_tau_d)
    canon_lbls, canon_sims = map_and_topk(dedup_lbls, dedup_embs, tau_m,
                                           best_k, mapping_log, arxiv_id)

    b1c_test_results.append({
        "arxiv_id":        arxiv_id,
        "title":           title,
        "n_agent_raw":     len(agent_lbls),
        "n_after_dedup":   len(dedup_lbls),
        "n_canonical":     len(canon_lbls),
        "canon_labels":    canon_lbls,
        "gold_labels":     gold_tags,
        "exact":           exact_prf(canon_lbls, gold_tags),
        "soft":            soft_prf(canon_lbls, gold_tags),
    })

# Save mapping log
Path("data/results/b1c_mappings.json").write_text(
    json.dumps(mapping_log, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Mapping log saved ({len(mapping_log)} entries).")

def macro_avg(key, subkey=None):
    vals = []
    for r in b1c_test_results:
        v = r[key] if subkey is None else r[key][subkey]
        vals.append(v)
    return round(float(np.mean(vals)), 4)


Mapping log saved (79 entries).


In [46]:
print("=" * 80)
print("CALIBRATION RESULTS  (Sensitivity Analysis)")
print(f"tau_m (frozen p90) = {tau_m:.4f}   k={best_k}   tau_d={best_tau_d}   prefix=clustering:")
print("=" * 80)
print(f"{'Metric':<28} {'B1 primary':>11} {'B1-hT-15':>11} {'B1c Exact':>11} {'B1s Cosine':>11}")
print(f"{'':28} {'n=22,hasTopic':>11} {'n=15,hT':>11} {'n=15,hT':>11} {'n=15,hT':>11}")
print("-" * 80)
print(f"{'Precision':<28} {'0.0762':>11} {macro_ht15('p'):>11.4f} {macro_avg('exact','p'):>11.4f} {macro_avg('soft','p'):>11.4f}")
print(f"{'Recall':<28} {'0.1508':>11} {macro_ht15('r'):>11.4f} {macro_avg('exact','r'):>11.4f} {macro_avg('soft','r'):>11.4f}")
print(f"{'F1':<28} {'0.0957':>11} {macro_ht15('f1'):>11.4f} {macro_avg('exact','f1'):>11.4f} {macro_avg('soft','f1'):>11.4f}")
print(f"{'Avg agent tags used':<28} {'5.3*':>11} {np.mean([r['n_agent'] for r in b1_ht15_results]):>11.1f} {macro_avg('n_canonical'):>11.2f} {macro_avg('n_canonical'):>11.2f}")
print("=" * 80)
print("* B1 primary uses hasTopic only on all 22 overlapping papers, with six bibliographic paper-title labels removed from the expert gold set.")
print()
print("B1-hT-15  : uncalibrated hasTopic-only baseline (paper-title tags removed, no mapping)")
print("B1c Exact : calibrated exact match into 49-label expert ontology")
print("B1s Cosine: cosine proximity upper bound — NOT semantic accuracy (see footnotes)")
print()
print("NOTE: B1c and B1s are exploratory held-out sensitivity analyses on 15 test papers.")
print("      They are NOT primary results. B1 (n=22) remains the primary reported metric.")

# Per-paper breakdown
print("\n--- Per-paper detail (test set, sorted by B1c F1) ---")
print(f"{'arXiv':>12}  {'B1-hT F1':>9}  {'B1c F1':>9}  {'B1s F1':>9}  {'n_hT':>5}  {'n_can':>6}  {'n_gold':>6}")
print("-" * 70)
hT15_by_id = {r['arxiv_id']: r for r in b1_ht15_results}
for r in sorted(b1c_test_results, key=lambda x: x['exact']['f1']):
    ht = hT15_by_id.get(r['arxiv_id'], {})
    ht_f1 = ht.get('metrics', {}).get('f1', 0.0)
    print(f"{r['arxiv_id']:>12}  {ht_f1:>9.4f}  {r['exact']['f1']:>9.4f}  {r['soft']['f1']:>9.4f}  "
          f"{ht.get('n_agent',0):>5}  {r['n_canonical']:>6}  {r['exact']['n_gold']:>6}")


CALIBRATION RESULTS  (Sensitivity Analysis)
tau_m (frozen p90) = 0.7361   k=4   tau_d=0.85   prefix=clustering:
Metric                        B1 primary    B1-hT-15   B1c Exact  B1s Cosine
                             n=22,hasTopic     n=15,hT     n=15,hT     n=15,hT
--------------------------------------------------------------------------------
Precision                         0.0762      0.1006      0.3389      0.8443
Recall                            0.1508      0.1989      0.4756      0.8811
F1                                0.0957      0.1256      0.3831      0.8614
Avg agent tags used                 5.3*         5.3        3.80        3.80
* B1 primary uses hasTopic only on all 22 overlapping papers, with six bibliographic paper-title labels removed from the expert gold set.

B1-hT-15  : uncalibrated hasTopic-only baseline (paper-title tags removed, no mapping)
B1c Exact : calibrated exact match into 49-label expert ontology
B1s Cosine: cosine proximity upper bound — NOT seman

In [47]:
print("=" * 65)
print("SUPPLEMENTARY SENSITIVITY CHECKS (binary soft-F1 at fixed thresholds)")
print("These are NOT used to select the best threshold.")
print("=" * 65)

def soft_prf_binary(pred_labels, gold_labels, threshold):
    if not pred_labels or not gold_labels:
        return dict(p=0.0, r=0.0, f1=0.0)
    E_pred = encode_labels(pred_labels)
    E_gold = encode_labels(gold_labels)
    sim    = E_pred @ E_gold.T
    # Binary: each pred earns 1 if its best gold sim >= threshold
    soft_p = float((sim.max(axis=1) >= threshold).mean())
    soft_r = float((sim.max(axis=0) >= threshold).mean())
    soft_f = 2*soft_p*soft_r / (soft_p+soft_r) if (soft_p+soft_r) > 0 else 0.0
    return dict(p=round(soft_p,4), r=round(soft_r,4), f1=round(soft_f,4))

for thresh in [0.75, 0.80]:
    p_vals, r_vals, f_vals = [], [], []
    for r in b1c_test_results:
        m = soft_prf_binary(r["canon_labels"], r["gold_labels"], thresh)
        p_vals.append(m["p"]); r_vals.append(m["r"]); f_vals.append(m["f1"])
    print(f"  threshold={thresh:.2f}  P={np.mean(p_vals):.4f}  "
          f"R={np.mean(r_vals):.4f}  F1={np.mean(f_vals):.4f}")
print()
print("Primary soft-F1 (continuous, no threshold):",
      f"P={macro_avg('soft','p'):.4f}  R={macro_avg('soft','r'):.4f}  F1={macro_avg('soft','f1'):.4f}")


SUPPLEMENTARY SENSITIVITY CHECKS (binary soft-F1 at fixed thresholds)
These are NOT used to select the best threshold.
  threshold=0.75  P=0.6222  R=0.6822  F1=0.6289
  threshold=0.80  P=0.5222  R=0.6467  F1=0.5587

Primary soft-F1 (continuous, no threshold): P=0.8443  R=0.8811  F1=0.8614


In [48]:
final_output = {
    "label": "B1c/B1s Sensitivity Analysis — Secondary Result",
    "tau_m_frozen":  tau_m,
    "best_k":        best_k,
    "best_tau_d":    best_tau_d,
    "n_dev":         len(dev_ids),
    "n_test":        len(test_ids),
    "dev_ids":       dev_ids,
    "test_ids":      test_ids,
    "B1c_exact_test": {
        "precision": macro_avg("exact", "p"),
        "recall":    macro_avg("exact", "r"),
        "f1":        macro_avg("exact", "f1"),
    },
    "B1s_soft_test": {
        "precision": macro_avg("soft", "p"),
        "recall":    macro_avg("soft", "r"),
        "f1":        macro_avg("soft", "f1"),
    },
    "per_paper": b1c_test_results,
}

out_path = Path("data/results/b1c_test_results.json")
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(final_output, f, indent=2, ensure_ascii=False)
print(f"Results saved -> {out_path}")


Results saved -> data\results\b1c_test_results.json
